# Capítulo 2 — Projetando sistemas de agentes

**Livro:** começa na página **43** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import sys
sys.path.insert(0, "../comum")  # ia.py: modelos via OpenRouter, com controle de gasto
from ia import *
import json, re, time, random
import numpy as np
print("Pronto. Modelo padrão:", MODELO)

## Our First Agent System · PDF p. 43

#### Exercício: Seu primeiro agente com ferramentas

Com as ferramentas `consultar_preco` e `status_pedido`, escreva o laço do agente: chame o modelo com `tools`; se ele pedir ferramentas, execute, devolva o resultado como mensagem `tool` e repita; pare quando vier texto (no máximo 5 passos). Teste com "quanto custa um teclado e onde está o pedido A100?".

In [ ]:
CATALOGO = {"teclado": 199.9, "mouse": 89.9, "monitor": 1299.0, "cadeira": 899.0}
PEDIDOS = {"A100": {"status": "enviado", "previsao": "2025-06-12"}, "A101": {"status": "em separação", "previsao": "2025-06-15"}}
def consultar_preco(produto: str) -> str:
    """Devolve o preço de um produto do catálogo."""
    p = CATALOGO.get(produto.lower())
    return f"{produto}: R$ {p:.2f}" if p else f"{produto} não está no catálogo"
def status_pedido(pedido_id: str) -> str:
    """Devolve o status e a previsão de entrega de um pedido."""
    p = PEDIDOS.get(pedido_id.upper())
    return json.dumps(p, ensure_ascii=False) if p else "pedido não encontrado"
FUNCOES = {"consultar_preco": consultar_preco, "status_pedido": status_pedido}
TOOLS = [{"type": "function", "function": {"name": "consultar_preco", "description": "Preço de um produto do catálogo",
          "parameters": {"type": "object", "properties": {"produto": {"type": "string"}}, "required": ["produto"]}}},
         {"type": "function", "function": {"name": "status_pedido", "description": "Status e previsão de entrega de um pedido",
          "parameters": {"type": "object", "properties": {"pedido_id": {"type": "string"}}, "required": ["pedido_id"]}}}]
def agente(pergunta, max_passos=5):
    msgs = [{"role": "system", "content": "Você atende uma loja. Use as ferramentas; não invente dados."},
            {"role": "user", "content": pergunta}]
    for _ in range(max_passos):
        r = conversar(msgs, tools=TOOLS).choices[0].message
        if not r.tool_calls: return r.content
        msgs.append(r)
        for c in r.tool_calls:
            res = FUNCOES[c.function.name](**json.loads(c.function.arguments)); print("🔧", c.function.name, "->", res)
            msgs.append({"role": "tool", "tool_call_id": c.id, "content": res})
    return "Parei: limite de passos."
print(agente("Quanto custa um teclado e onde está o pedido A100?"))

## Tools · PDF p. 54

#### Exercício: Esquema da ferramenta gerado da função

Escreva `esquema(funcao)` que usa `inspect.signature` e a *docstring* para gerar automaticamente o dicionário de ferramenta no formato da OpenAI (tipos `str`→string, `int`→integer, `float`→number, `bool`→boolean). Teste com as duas funções da loja.

In [ ]:
CATALOGO = {"teclado": 199.9, "mouse": 89.9, "monitor": 1299.0, "cadeira": 899.0}
PEDIDOS = {"A100": {"status": "enviado", "previsao": "2025-06-12"}, "A101": {"status": "em separação", "previsao": "2025-06-15"}}
def consultar_preco(produto: str) -> str:
    """Devolve o preço de um produto do catálogo."""
    p = CATALOGO.get(produto.lower())
    return f"{produto}: R$ {p:.2f}" if p else f"{produto} não está no catálogo"
def status_pedido(pedido_id: str) -> str:
    """Devolve o status e a previsão de entrega de um pedido."""
    p = PEDIDOS.get(pedido_id.upper())
    return json.dumps(p, ensure_ascii=False) if p else "pedido não encontrado"
FUNCOES = {"consultar_preco": consultar_preco, "status_pedido": status_pedido}
TOOLS = [{"type": "function", "function": {"name": "consultar_preco", "description": "Preço de um produto do catálogo",
          "parameters": {"type": "object", "properties": {"produto": {"type": "string"}}, "required": ["produto"]}}},
         {"type": "function", "function": {"name": "status_pedido", "description": "Status e previsão de entrega de um pedido",
          "parameters": {"type": "object", "properties": {"pedido_id": {"type": "string"}}, "required": ["pedido_id"]}}}]
import inspect
TIPOS = {str: "string", int: "integer", float: "number", bool: "boolean"}
def esquema(f):
    sig = inspect.signature(f)
    props = {n: {"type": TIPOS.get(p.annotation, "string")} for n, p in sig.parameters.items()}
    obrig = [n for n, p in sig.parameters.items() if p.default is inspect.Parameter.empty]
    return {"type": "function", "function": {"name": f.__name__, "description": inspect.getdoc(f) or "",
            "parameters": {"type": "object", "properties": props, "required": obrig}}}
print(json.dumps(esquema(consultar_preco), ensure_ascii=False, indent=1))
print(esquema(status_pedido) == TOOLS[1] or "difere só na descrição")

## Memory · PDF p. 56

#### Exercício: Memória de conversa com limite de tokens

Escreva a classe `Memoria` que guarda as mensagens e, ao montar o contexto, mantém sempre a mensagem de sistema e só as mensagens **mais recentes** que cabem em `limite` tokens (`contar_tokens`). Teste com 30 mensagens.

In [ ]:
class Memoria:
    def __init__(self, sistema, limite=120):
        self.sistema, self.limite, self.msgs = {"role": "system", "content": sistema}, limite, []
    def adicionar(self, papel, texto): self.msgs.append({"role": papel, "content": texto})
    def contexto(self):
        escolhidas, total = [], contar_tokens(self.sistema["content"])
        for m in reversed(self.msgs):
            t = contar_tokens(m["content"])
            if total + t > self.limite: break
            escolhidas.append(m); total += t
        return [self.sistema] + escolhidas[::-1]
mem = Memoria("Você é um assistente.")
for i in range(30): mem.adicionar("user" if i % 2 == 0 else "assistant", f"mensagem número {i} com algum conteúdo")
ctx = mem.contexto(); print(len(ctx), "mensagens no contexto:", [m["content"][-2:] for m in ctx[1:]])

## Design Trade-Offs · PDF p. 59

#### Exercício: Latência de passos em série × paralelo

Um agente precisa de 4 chamadas de ferramenta independentes de 0,3 s cada, mais 2 chamadas ao modelo de 1,2 s. Simule com `time.sleep` e compare o tempo total em série e com as ferramentas em paralelo (`ThreadPoolExecutor`).

In [ ]:
from concurrent.futures import ThreadPoolExecutor
def ferramenta(i): time.sleep(0.3); return i
def modelo(): time.sleep(1.2)
t = time.perf_counter(); modelo(); [ferramenta(i) for i in range(4)]; modelo(); serie = time.perf_counter() - t
t = time.perf_counter(); modelo()
with ThreadPoolExecutor(4) as ex: list(ex.map(ferramenta, range(4)))
modelo(); paralelo = time.perf_counter() - t
print(f"série {serie:.2f}s | paralelo {paralelo:.2f}s")